In [6]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

# **Data Collection:**

In [7]:
# Load dataset
file_path = '/content/spam.csv'
dataset = pd.read_csv(file_path, encoding='latin-1')

In [8]:
# Retain only relevant columns
dataset = dataset[['v1', 'v2']]
dataset.columns = ['label', 'message']  # Rename for clarity

In [9]:
# Encode labels (ham = 0, spam = 1)
dataset['label'] = dataset['label'].map({'ham': 0, 'spam': 1})

# **Data Cleaning:**

In [38]:
df = pd.read_csv('/content/spam.csv', encoding='latin1')

In [39]:
df = df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'])

In [40]:
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

1. Lowercasing

In [41]:
df['v2'] = df['v2'].apply(lambda x: x.lower())

2. Removing punctuation and special characters

In [42]:
df['v2'] = df['v2'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

3. Tokenization (splitting text into words)

In [43]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [44]:
df['v2'] = df['v2'].apply(lambda x: word_tokenize(x))

4. Remove stop words:

In [45]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [46]:
stop_words = set(stopwords.words('english'))
df['v2'] = df['v2'].apply(lambda x: [word for word in x if word not in stop_words])

5. Stemming:

In [47]:
ps = PorterStemmer()
df['v2'] = df['v2'].apply(lambda x: [ps.stem(word) for word in x])

In [48]:
df.head()

,v1,v2
0,ham,"[go, jurong, point, crazi, avail, bugi, n, gre..."
1,ham,"[ok, lar, joke, wif, u, oni]"
2,spam,"[free, entri, 2, wkli, comp, win, fa, cup, fin..."
3,ham,"[u, dun, say, earli, hor, u, c, alreadi, say]"
4,ham,"[nah, dont, think, goe, usf, live, around, tho..."


**Label Encoding:**

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

In [50]:
# 2. Label Encoding (ham -> 0, spam -> 1)
label_encoder = LabelEncoder()
df['v1'] = label_encoder.fit_transform(df['v1'])

In [51]:
# 3. Join the tokens back into a single string for TF-IDF processing
df['v2'] = df['v2'].apply(lambda x: ' '.join(x))

**TF-IDF Vectorization:**

In [52]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['v2']).toarray()

In [53]:
# Labels (ham=0, spam=1)
y = df['v1'].values

# Check the shape of the feature matrix
print("Feature matrix shape:", X.shape)

Feature matrix shape: (5572, 5000)


In [54]:
df.head()

,v1,v2
0,0,go jurong point crazi avail bugi n great world...
1,0,ok lar joke wif u oni
2,1,free entri 2 wkli comp win fa cup final tkt 21...
3,0,u dun say earli hor u c alreadi say
4,0,nah dont think goe usf live around though


# **Data Preprocessing:**

In [10]:
# Clean the text (remove special characters, numbers, and lowercase)
def clean_text(text):
    text = re.sub(r'\W', ' ', text)  # Remove special characters
    text = re.sub(r'\s+', ' ', text)  # Remove extra spaces
    text = re.sub(r'\d', '', text)  # Remove numbers
    return text.strip().lower()

In [11]:
dataset['message'] = dataset['message'].apply(clean_text)

In [12]:
dataset['message']

,message
0,go until jurong point crazy available only in ...
1,ok lar joking wif u oni
2,free entry in a wkly comp to win fa cup final...
3,u dun say so early hor u c already then say
4,nah i don t think he goes to usf he lives arou...
...,...
5567,this is the nd time we have tried contact u u...
5568,will ì_ b going to esplanade fr home
5569,pity was in mood for that so any other suggest...
5570,the guy did some bitching but i acted like i d...


In [13]:
# Split dataset into training and testing sets
X = dataset['message']
y = dataset['label']

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data preprocessing complete.")

Data preprocessing complete.


# **Feature Extraction**

**Bag of Words:**

In [15]:
from sklearn.feature_extraction.text import CountVectorizer

In [16]:
vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 2))  # Use unigrams and bigrams
X_train_features = vectorizer.fit_transform(X_train).toarray()

In [17]:
X_test_features = vectorizer.transform(X_test).toarray()

print("Feature extraction complete.")

Feature extraction complete.


# **Model Development:**

In [18]:
from keras.models import Sequential
from keras.layers import Dense, Dropout, Conv1D, GlobalMaxPooling1D, Embedding

In [19]:
embedding_dim = 50
vocab_size = X_train_features.shape[1]

In [20]:
# Build the CNN model
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=vocab_size),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(units=64, activation='relu'),
    Dropout(0.5),
    Dense(units=1, activation='sigmoid')  # Binary classification
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [21]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d (Conv1D)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d                 │ ?                           │     0 (unbuilt) │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [33]:
# Train the CNN model
history = model.fit(
    X_train_features,
    y_train,
    epochs=30,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

Epoch 1/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 56s 898ms/step - accuracy: 0.8578 - loss: 0.3303 - val_accuracy: 0.8565 - val_loss: 0.3461
Epoch 2/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 81s 881ms/step - accuracy: 0.8635 - loss: 0.3388 - val_accuracy: 0.8565 - val_loss: 0.3329
Epoch 3/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 84s 913ms/step - accuracy: 0.8717 - loss: 0.3139 - val_accuracy: 0.8565 - val_loss: 0.3327
Epoch 4/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 79s 872ms/step - accuracy: 0.8576 - loss: 0.3147 - val_accuracy: 0.8565 - val_loss: 0.3330
Epoch 5/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 84s 906ms/step - accuracy: 0.8698 - loss: 0.3178 - val_accuracy: 0.8565 - val_loss: 0.3337
Epoch 6/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 55s 871ms/step - accuracy: 0.8590 - loss: 0.3238 - val_accuracy: 0.8565 - val_loss: 0.3328
Epoch 7/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 84s 904ms/step - accuracy: 0.8747 - loss: 0.2993 - val_accuracy: 0.8565 - val_loss: 0.3328
Epoch 8/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 80s 876ms/step - accuracy: 0.8630 - loss: 0.3107 - val_accu

# **Model Evaluation**

In [34]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [35]:
# Evaluate the model
y_pred = (model.predict(X_test_features) > 0.5).astype("int32")

35/35 ━━━━━━━━━━━━━━━━━━━━ 4s 107ms/step


In [36]:
# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [37]:
print(f"Accuracy: {accuracy:.2f}")

Accuracy: 0.87


# **Sand Cat Swarm Optimization (SCSO)**

In [29]:
# Define the Sand Cat Swarm Optimization Algorithm
class SCSO:
    def __init__(self, fitness_function, n_agents, n_dims, lower_bound, upper_bound, max_iter):
        self.fitness_function = fitness_function  # Function to minimize
        self.n_agents = n_agents  # Number of agents (sand cats)
        self.n_dims = n_dims  # Dimensionality of the solution (weights in CNN)
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound
        self.max_iter = max_iter

        # Initialize the population
        self.population = np.random.uniform(lower_bound, upper_bound, (n_agents, n_dims))
        self.best_agent = None
        self.best_fitness = np.inf

    def optimize(self):
        for iteration in range(self.max_iter):
            for i in range(self.n_agents):
                fitness = self.fitness_function(self.population[i])

                # Update the best solution
                if fitness < self.best_fitness:
                    self.best_fitness = fitness
                    self.best_agent = self.population[i]

            # Update positions based on the exploration and exploitation phases
            for i in range(self.n_agents):
                r = 2 - (2 * iteration / self.max_iter)  # Exploration factor decreases over iterations
                random_factor = np.random.uniform(-1, 1, self.n_dims)

                # Compute new position
                self.population[i] += r * random_factor * (self.best_agent - self.population[i])

                # Clip to boundary
                self.population[i] = np.clip(self.population[i], self.lower_bound, self.upper_bound)

            print(f"Iteration {iteration + 1}/{self.max_iter}, Best Fitness: {self.best_fitness}")

        return self.best_agent, self.best_fitness

# Define the fitness function for the CNN
def cnn_fitness(weights):
    # Placeholder: Replace this with actual training logic using weights
    # E.g., apply weights to CNN and compute validation loss
    validation_loss = np.sum(weights**2)  # Example: L2 loss (to minimize)
    return validation_loss

In [30]:
# Apply SCSO to optimize weights
n_agents = 10  # Number of sand cats
n_dims = sum([np.prod(layer.shape) for layer in model.trainable_weights])  # Total weights in CNN
lower_bound = -1  # Min weight value
upper_bound = 1   # Max weight value
max_iter = 50  # Number of iterations

In [31]:
# Initialize and run SCSO
scso = SCSO(fitness_function=cnn_fitness, n_agents=n_agents, n_dims=n_dims,
            lower_bound=lower_bound, upper_bound=upper_bound, max_iter=max_iter)
best_weights, best_fitness = scso.optimize()

Iteration 1/50, Best Fitness: 87590.66292383354
Iteration 2/50, Best Fitness: 87590.66292383354
Iteration 3/50, Best Fitness: 87590.66292383354
Iteration 4/50, Best Fitness: 87590.66292383354
Iteration 5/50, Best Fitness: 87590.66292383354
Iteration 6/50, Best Fitness: 87590.66292383354
Iteration 7/50, Best Fitness: 87590.66292383354
Iteration 8/50, Best Fitness: 87590.66292383354
Iteration 9/50, Best Fitness: 87590.66292383354
Iteration 10/50, Best Fitness: 87590.66292383354
Iteration 11/50, Best Fitness: 87590.66292383354
Iteration 12/50, Best Fitness: 87590.66292383354
Iteration 13/50, Best Fitness: 87590.66292383354
Iteration 14/50, Best Fitness: 87590.66292383354
Iteration 15/50, Best Fitness: 87590.66292383354
Iteration 16/50, Best Fitness: 87590.66292383354
Iteration 17/50, Best Fitness: 87590.66292383354
Iteration 18/50, Best Fitness: 87590.66292383354
Iteration 19/50, Best Fitness: 87590.66292383354
Iteration 20/50, Best Fitness: 87590.66292383354
Iteration 21/50, Best Fitness

In [32]:
print("Optimization Complete")
print(f"Best Fitness: {best_fitness}")

Optimization Complete
Best Fitness: 87590.66292383354
